In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense, TimeDistributed, Bidirectional
from keras.optimizers import Adam, AdamW
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import load_model
import pickle

2024-12-17 19:14:48.364498: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-17 19:14:48.371836: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-12-17 19:14:48.392362: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1734452088.429474    3831 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1734452088.441183    3831 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-17 19:14:48.476367: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

In [2]:

# Загружаем данные из CSV
data = pd.read_csv('clean_wordsNO.csv', names=['word', 'transcription'])

# Создание отображения символов в индексы и наоборот
char_to_index = {}
index_to_char = {}

def create_mapping(data):
    global char_to_index, index_to_char
    unique_chars = set(''.join(data['word']) + ''.join(data['transcription']))
    char_to_index = {char: idx + 1 for idx, char in enumerate(unique_chars)}
    index_to_char = {idx + 1: char for idx, char in enumerate(unique_chars)}
    char_to_index['<PAD>'] = 0
    index_to_char[0] = '<PAD>'

create_mapping(data)

# Кодирование строк в последовательности индексов
def encode_sequence(sequence, max_length):
    return [char_to_index[char] for char in sequence] + [char_to_index['<PAD>']] * (max_length - len(sequence))

# Определение максимальной длины последовательностей
max_word_length = max(data['word'].apply(len))
max_transcription_length = max(data['transcription'].apply(len))
max_sequence_length = max(max_word_length, max_transcription_length)

# Кодируем входные и выходные данные
X = np.array([encode_sequence(word, max_sequence_length) for word in data['word']])
y = np.array([encode_sequence(transcription, max_sequence_length) for transcription in data['transcription']])

# Преобразуем выходные данные в one-hot представление
y = tf.keras.utils.to_categorical(y, num_classes=len(char_to_index))

# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создание модели
model = Sequential([
    Embedding(input_dim=len(char_to_index), output_dim=128, input_length=max_sequence_length),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64, return_sequences=True)),
    TimeDistributed(Dense(256, activation='relu')),
    Dropout(0.3),
    TimeDistributed(Dense(len(char_to_index), activation='softmax'))
])

# Компиляция модели
model.compile(optimizer=AdamW(learning_rate=0.005), loss='categorical_crossentropy', metrics=['accuracy'])

# EarlyStopping Callback
early_stopping = EarlyStopping(
    monitor='val_loss',  # Метрика, которую отслеживаем
    patience=3,          # Количество эпох без улучшений, после которых остановится обучение
    restore_best_weights=True  # Возвращает веса с лучшим val_loss
)

# Обучение модели с использованием EarlyStopping
model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    batch_size=15,
    epochs=50,  # Устанавливаем максимальное количество эпох
    callbacks=[early_stopping]  # Передаём Callback
)

# Функция для предсказания транскрипции
def predict_transcription(word):
    encoded_word = encode_sequence(word, max_sequence_length)
    encoded_word = np.expand_dims(encoded_word, axis=0)
    predicted_indices = model.predict(encoded_word)
    predicted_indices = np.argmax(predicted_indices, axis=-1)[0]
    transcription = ''.join(index_to_char[idx] for idx in predicted_indices if idx != 0)
    return transcription.strip()

# сохранение модели
model.save('work.keras')
# сохранение маппинга
with open('work.pkl', 'wb') as f:
    pickle.dump({'char_to_index': char_to_index, 'index_to_char': index_to_char}, f)


# Пример использования
example_word = "пример"
predicted_transcription = predict_transcription(example_word)
print(f"'{example_word}' -> '{predicted_transcription}'")

/home/lamia/SchoolX/ipa/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2024-12-09 16:43:30.340960: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 111s 133ms/step - accuracy: 0.8436 - loss: 0.6556 - val_accuracy: 0.9196 - val_loss: 0.2647
Epoch 2/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 92s 128ms/step - accuracy: 0.9188 - loss: 0.2684 - val_accuracy: 0.9380 - val_loss: 0.1965
Epoch 3/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 92s 127ms/step - accuracy: 0.9350 - loss: 0.2068 - val_accuracy: 0.9469 - val_loss: 0.1688
Epoch 4/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 92s 127ms/step - accuracy: 0.9432 - loss: 0.1795 - val_accuracy: 0.9501 - val_loss: 0.1618
Epoch 5/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 94s 130ms/step - accuracy: 0.9478 - loss: 0.1648 - val_accuracy: 0.9548 - val_loss: 0.1428
Epoch 6/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 93s 128ms/step - accuracy: 0.9537 - loss: 0.1452 - val_accuracy: 0.9588 - val_loss: 0.1377
Epoch 7/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 92s 127ms/step - accuracy: 0.9572 - loss: 0.1339 - val_accuracy: 0.9589 - val_loss: 0.1381
Epoch 8/50
724/724 ━━━━━━━━━━━━━━━━━━━━ 91s 126ms/step - accuracy: 0.9587 - loss: 

In [10]:
import pickle

with open('work.pkl', 'rb') as f:
    mapping = pickle.load(f)
char_to_index = mapping['char_to_index']
index_to_char = mapping['index_to_char']

# Создание обратного маппинга
index_to_char = {index: char for char, index in char_to_index.items()}

# Загрузка модели
model = load_model('work.keras')
max_sequence_length = 15
# Функция для кодирования последовательности
def encode_sequence(sequence, max_length):
    return [char_to_index.get(char, 0) for char in sequence] + [char_to_index['<PAD>']] * (max_length - len(sequence))

# Функция для предсказания транскрипции
def predict_transcription(word, max_sequence_length):
    encoded_word = encode_sequence(word, max_sequence_length)
    encoded_word = np.expand_dims(encoded_word, axis=0)
    predicted_indices = model.predict(encoded_word)
    predicted_indices = np.argmax(predicted_indices, axis=-1)[0]
    transcription = ''.join(index_to_char[idx] for idx in predicted_indices if idx != 0)
    return transcription.strip()

# Пример использования
example_word = "пример"
predicted_transcription = predict_transcription(example_word, max_sequence_length)
print(f"'{example_word}' -> '{predicted_transcription}'")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
'пример' -> 'prʲimʲɪr'


In [11]:
example_word = "пингвин"
predicted_transcription = predict_transcription(example_word, max_sequence_length)
print(f"'{example_word}' -> '{predicted_transcription}'")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
'пингвин' -> 'pʲɪnɡvʲɪn'


In [13]:
import re

# Функция для очистки текста от знаков препинания
def clean_text(text):
    # Оставляем только буквы и пробелы, удаляя знаки препинания
    return re.sub(r'[^\w\s]', '', text)

# Текст для предсказания
text = """Добрый день, извините, Вы не могли бы мне помочь? Кажется, я заблудился. 
Интернет перестал ловить и не загружается карта. Я не знаю, где сейчас нахожусь.

Добрый день, не волнуйтесь, всё хорошо, я Вам помогу. Мы находимся на Ворошиловском проспекте. А Вам куда нужно?

Мне нужно добраться до студенческого общежития номер десять Донского государственного технического университета.

Это общежитие находится возле главного корпуса ДГТУ на площади Гагарина один. Вам нужно идти прямо, никуда не сворачивать. Прямо перед Вами будет площадь Гагарина с фонтаном и здание университета. Обогните главный корпус слева и увидите высокое круглое здание. Это и есть общежитие номер десять.

Большое спасибо за помощь!"""



# Шаг 1: Очистка текста от знаков препинания
cleaned_text = clean_text(text)


# Шаг 2: Разделение на слова и предсказание транскрипции
words = cleaned_text.split()
predictions = [predict_transcription(word, max_sequence_length) for word in words]


# Шаг 3: Объединение результатов в строку
predicted_text = ' '.join(predictions)

# Выводим только транскрипцию текста
print(predicted_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step


KeyboardInterrupt: 

In [ ]:
import re

# Функция для очистки текста от знаков препинания
def clean_text(text):
    # Оставляем только буквы и пробелы, удаляя знаки препинания
    return re.sub(r'[^ws]', '', text)

# Пример функции predict_transcription (замените на вашу реализацию)
def predict_transcription(word, max_sequence_length):
    # Здесь должна быть ваша логика предсказания транскрипции
    # Для примера просто вернем слово в верхнем регистре
    return word.upper()

# Текст для предсказания
text = """Добрый день, извините, Вы не могли бы мне помочь? Кажется, я заблудился. 
Интернет перестал ловить и не загружается карта. Я не знаю, где сейчас нахожусь.

Добрый день, не волнуйтесь, всё хорошо, я Вам помогу. Мы находимся на Ворошиловском проспекте. А Вам куда нужно?

Мне нужно добраться до студенческого общежития номер десять Донского государственного технического университета.

Это общежитие находится возле главного корпуса ДГТУ на площади Гагарина один. Вам нужно идти прямо, никуда не сворачивать. Прямо перед Вами будет площадь Гагарина с фонтаном и здание университета. Обогните главный корпус слева и увидите высокое круглое здание. Это и есть общежитие номер десять.

Большое спасибо за помощь!"""

# Шаг 1: Очистка текста от знаков препинания
cleaned_text = clean_text(text)

# Шаг 2: Разделение на слова и предсказание транскрипции
words = cleaned_text.split()
predictions = [predict_transcription(word, max_sequence_length) for word in words,]

# Шаг 3: Объединение результатов в строку
predicted_text = ' '.join(predictions)

# Выводим только транскрипцию текста
print(predicted_text)


In [2]:
import numpy as np
from tensorflow.keras.models import load_model
import pickle

# Загрузка модели
model = load_model('work.keras')

# Определение функции для кодирования последовательности
def encode_sequence(sequence, max_length):
    return [char_to_index.get(char, 0) for char in sequence] + [char_to_index['<PAD>']] * (max_length - len(sequence))

# Определение функции для предсказания транскрипции
def predict_transcription(word, max_sequence_length):
    encoded_word = encode_sequence(word, max_sequence_length)
    encoded_word = np.expand_dims(encoded_word, axis=0)
    predicted_indices = model.predict(encoded_word)
    predicted_indices = np.argmax(predicted_indices, axis=-1)[0]
    transcription = ''.join(index_to_char[idx] for idx in predicted_indices if idx != 0)
    return transcription.strip()

# Укажите максимальную длину последовательностей (вы должны знать ее из обучения)
max_sequence_length = 15000

# Пример использования
example_word = "пример"
predicted_transcription = predict_transcription(example_word, max_sequence_length)
print(f"'{example_word}' -> '{predicted_transcription}'")


2024-12-17 19:14:58.875490: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


NameError: name 'char_to_index' is not defined

In [3]:
# Функция для предсказания транскрипции
def predict_transcription(word):
    # Используем глобальную переменную max_sequence_length
    encoded_word = encode_sequence(word, max_sequence_length)
    encoded_word = np.expand_dims(encoded_word, axis=0)
    predicted_indices = model.predict(encoded_word)
    predicted_indices = np.argmax(predicted_indices, axis=-1)[0]
    transcription = ''.join(index_to_char[idx] for idx in predicted_indices if idx != 0)
    return transcription.strip()

# Пример использования
example_word = "пингвин"
predicted_transcription = predict_transcription(example_word)
print(f"'{example_word}' -> '{predicted_transcription}'")


NameError: name 'char_to_index' is not defined

In [14]:
import re

# Функция для очистки текста от знаков препинания
def clean_text(text):
    # Оставляем только буквы и пробелы, удаляя знаки препинания
    return re.sub(r'[^\w\s]', '', text)

# Текст для предсказания
text = """Добрый день, извините, Вы не могли бы мне помочь? Кажется, я заблудился. 
Интернет перестал ловить и не загружается карта. Я не знаю, где сейчас нахожусь.

Добрый день, не волнуйтесь, всё хорошо, я Вам помогу. Мы находимся на Ворошиловском проспекте. А Вам куда нужно?

Мне нужно добраться до студенческого общежития номер десять Донского государственного технического университета.

Это общежитие находится возле главного корпуса ДГТУ на площади Гагарина один. Вам нужно идти прямо, никуда не сворачивать. Прямо перед Вами будет площадь Гагарина с фонтаном и здание университета. Обогните главный корпус слева и увидите высокое круглое здание. Это и есть общежитие номер десять.

Большое спасибо за помощь!"""

max_sequence_length = 15  
# Шаг 1: Очистка текста от знаков препинания
cleaned_text = clean_text(text)

# Шаг 2: Разделение на слова и предсказание транскрипции
words = cleaned_text.split()
predictions = [predict_transcription(word) for word in words]


# Шаг 3: Объединение результатов в строку
predicted_text = ' '.join(predictions)

# Выводим только транскрипцию текста
print(predicted_text)


TypeError: predict_transcription() missing 1 required positional argument: 'max_sequence_length'